# **BindScore** — pursuing computationally cheap protein-protein binding energy estimation.

**Introduction.**

Predicting protein–protein binding affinity from structure is a long-standing problem in computational biology and chemistry. It has been addressed by a wide spectrum of tools, ranging from ones based on established Molecular Dynamis (MD) methods (free-energy perturbation, MM-PBSA,etc) through empirical scorers (like FoldX, PRODIGY, Rosetta InterfaceAnalyzer and others) to a recent wave of modern deep-learning predictors (AlphaFold-Multimer-derived models, DeepDDG, etc). Each family trades accuracy against speed, transparency, and the amount of structural input it needs — FEP costs hours of MD per complex but reaches sub-kcal/mol accuracy on favourable cases, while empirical scorers return a number in seconds from a single static structure at the cost of a coarser treatment of entropy and solvation. BindScore is a from-scratch implementation in the spirit of the fast empirical category, written as a course project to make every scoring term physically explicit rather than to compete with production tools.


BindScore is a from-scratch static-structure scorer that estimates the binding free energy between two entities (protein chains, small molecules, or metals) inside a single PDB file. The baseline goal of the package was to achieve a running timeframe measured in seconds. Therefore no molecular-dynamics engine were implemented in the workingloop. 

From the beginning, a clear separation was made between Entropical and Enthalpical sides of the project, since binding Enthalpy was a subject to which we were more aquainted from our current level of studies. 

**Enthalpy scoring**
 
It detects atom-atom contacts at the interface, classifies each one into a physical interaction type (hydrogen bond, salt bridge, hydrophobic contact, π–π stacking, halogen bond, dipole–dipole, disulfide, or metal coordination), and scores them with the analytic potential that fits that type. The enthalpic pipeline is complete and produces a per-interaction breakdown; the entropic side is left as a documented set of stubs for future work.

**A try in entropy scoring**

This notebook runs the BindScore pipeline end-to-end on a single PDB entry:

**Running pipeline**

1. Fetch the structure from RCSB database.
2. Parse it into a `Protein_Structure` object and inspect its contents.
3. Build an `Interaction` object between two chains and look at the contacts that were detected.
4. Score the enthalpic contribution.
5. Breakdown by interaction type and inspect.
6. Entropy section - to be implemented.
7. Combine everything into a final ΔG estimate — also left empty for the same reason.
8. Displaying reuslts in Streamlit

Authors: Bogdan Filipchuk, Tuna Karasu, Andrey Babenko, Maximus van den Bogaard (EPFL).

## 1. Setup

BindScore is split across two subpackages — `pdb_file_treatment` (download + parsing) and `scoring` (interaction detection + enthalpy). The next cell makes sure both are importable regardless of where the notebook is launched from.

In [13]:
import sys
import pathlib

# Resolve repository root from the notebook location, then expose the two
# subpackages we need. This keeps the notebook runnable from anywhere.
ROOT = pathlib.Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pdb_file_treatment').is_dir() and (candidate / 'scoring').is_dir():
        ROOT = candidate
        break

for sub in ('pdb_file_treatment', 'scoring'):
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

print('Repository root :', ROOT)
print('pdb_file_treatment present :', (ROOT / 'pdb_file_treatment').is_dir())
print('scoring            present :', (ROOT / 'scoring').is_dir())

Repository root : c:\Users\bogfi\VScode\BindScore Project\Notebooks
pdb_file_treatment present : False
scoring            present : False


In [14]:
from pdb_utils_fetch    import fetch_pdb_data
from pdb_utils_protein  import Protein_Structure
from pdb_utils_inter    import Interaction
from pdb_utils_enthalpy import binding_enthalpy, interaction_energy

ModuleNotFoundError: No module named 'pdb_utils_fetch'

## 2. Choose a target

We use **1BRS** (Barnase–Barstar) as the running example. Barnase is an RNase from *Bacillus amyloliquefaciens* and Barstar is its intracellular inhibitor; the complex is one of the most-studied protein–protein interfaces in the literature and has an experimental K_d ≈ 10⁻¹⁴ M (Schreiber & Fersht, 1993).

Change `PDB_ID`, `CHAIN_A`, `CHAIN_B` to point the rest of the notebook at a different complex.

In [ ]:
PDB_ID    = '1BRS'
CHAIN_A   = 'A'     # barnase
CHAIN_B   = 'D'     # barstar
THRESHOLD = 5.0     # Å, distance cutoff for atom-atom contacts

## 3. Fetch and clean the structure

`fetch_pdb_data` accepts either a 4-character RCSB ID or a local `.pdb` / `.cif` path. CIF input is converted to PDB on the fly via `gemmi`.

If the entry is an NMR ensemble we trim it to the first model — the scorer is single-frame and would otherwise score every conformer.

In [ ]:
pdb_data = fetch_pdb_data(PDB_ID)

# Trim NMR ensembles to model 1 — the scorer is single-frame.
if 'MODEL' in pdb_data:
    pdb_data = pdb_data.split('ENDMDL')[0] + 'ENDMDL\n'

print(f'Received {len(pdb_data):,} characters of PDB text.')

## 4. Parse into a `Protein_Structure`

`Protein_Structure` parses every `ATOM` / `HETATM` line into a dictionary and gives back named accessors for the things we want later: chains, side chains, backbone, aromatic rings (with centroid and normal already computed), and small molecules / metals.

In [ ]:
protein = Protein_Structure(pdb_data)

print('Chains                :', protein.chains())
print('Small molecules / HET :', protein.small_molecules())
print('Total atoms parsed    :', len(protein.all_atoms))
print('Backbone atoms        :', len(protein.backbone()))
print('Side-chain atoms      :', len(protein.side_chain()))
print('Ring atoms            :', len(protein.ring_atoms()))

## 5. Detect contacts between the two chains

The `Interaction` constructor iterates over every (atom in entity 1) × (atom in entity 2) pair, keeps those within the threshold, and classifies each one. The result is a list of contact dicts on `inter.interactions`:

```python
{'atom1': {...}, 'atom2': {...}, 'distance': 2.93, 'type': 'hydrogen_bond'}
```

Salt bridges and π–π contacts are deduplicated per residue / ring pair, so a Glu–Arg salt bridge is reported once even if several of its donor/acceptor atoms happen to fall inside the cutoff.

In [ ]:
inter = Interaction(protein, CHAIN_A, CHAIN_B, threshold=THRESHOLD)

print(f'{len(inter.interactions):>4d} atom-pair contacts within {THRESHOLD} Å '
      f'between chain {CHAIN_A} and chain {CHAIN_B}.')

In [ ]:
# Histogram of contacts by classified type.
from collections import Counter

counts = Counter(c['type'] or 'unclassified' for c in inter.interactions)
for itype, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f'  {itype:<22s} {n:>4d}')

In [ ]:
# A few representative classified contacts.
classified = [c for c in inter.interactions if c['type'] is not None][:5]

for c in classified:
    a1, a2 = c['atom1'], c['atom2']
    print(f"{c['type']:<20s}  "
          f"{a1['residue_name']}{a1['residue_seq']:>4s}.{a1['atom_name']:<4s}"
          f"  --  "
          f"{a2['residue_name']}{a2['residue_seq']:>4s}.{a2['atom_name']:<4s}"
          f"  d = {c['distance']:.2f} Å")

## 6. Score the enthalpy

`binding_enthalpy(inter)` walks the contact list, dispatches each contact to the potential matching its type, aggregates salt bridges per residue pair, adds a desolvation penalty for buried polar groups, and returns a breakdown dictionary.

Conventions:
- Energies are in **kJ/mol**.
- Negative = favourable; positive = unfavourable.
- The `desolvation` term is the cost of removing waters from polar groups that engage in interface H-bonds / salt bridges; it is the one positive term in the breakdown.

In [ ]:
breakdown = binding_enthalpy(inter)

print(f'\n--- Enthalpy breakdown — {PDB_ID} chains {CHAIN_A}/{CHAIN_B} ---')
for itype, energy in sorted(breakdown.items(), key=lambda kv: kv[1]):
    if itype == 'TOTAL':
        continue
    print(f'  {itype:<22s} {energy:>+9.2f} kJ/mol')
print('  ' + '-'*32)
print(f"  {'TOTAL ΔH':<22s} {breakdown['TOTAL']:>+9.2f} kJ/mol")

In [ ]:
# The strongest individual contacts (after binding_enthalpy() has populated
# the per-pair 'energy' field).
ranked = sorted(
    (c for c in inter.interactions if c.get('energy', 0.0) != 0.0),
    key=lambda c: c['energy'],
)[:8]

print(f"{'type':<20s}  {'residue 1':<14s}  {'residue 2':<14s}  {'d (Å)':>6s}  {'E (kJ/mol)':>12s}")
print('-' * 72)
for c in ranked:
    a1, a2 = c['atom1'], c['atom2']
    r1 = f"{a1['residue_name']}{a1['residue_seq']}.{a1['atom_name']}"
    r2 = f"{a2['residue_name']}{a2['residue_seq']}.{a2['atom_name']}"
    print(f"{c['type']:<20s}  {r1:<14s}  {r2:<14s}  {c['distance']:>6.2f}  {c['energy']:>+12.2f}")

### What just happened

Each interaction type was scored with the potential that physically fits it:

- Hydrogen bonds and hydrophobic contacts use a **Gaussian well** centered on the typical equilibrium distance (3.0 Å, depth 8 kJ/mol for H-bonds; 3.8 Å, depth 0.5 kJ/mol for hydrophobic).
- Salt bridges are aggregated per residue pair, then scored with a **screened Coulomb** law (ε_r = 50 to mimic the high local dielectric at a water-exposed interface; a hard floor of 2.8 Å on the distance keeps the energy finite).
- π–π stacking is scored with a **Lennard-Jones-like well** whose depth and equilibrium distance depend on the angle between the two ring normals (parallel face-to-face is stronger than edge-to-face or T-shape).
- Halogen bonds, disulfide bonds, and metal coordination use **LJ wells** with type-specific depths.
- Dipole–dipole contacts use a fraction of the screened Coulomb between the two **atomic partial charges**, with the sign of the contribution set by the sign of the charge product.
- **Desolvation** is treated as a sum of one water H-bond enthalpy (4.25 kJ/mol) per buried polar contact, capped by the H-bond capacity of the buried atom (so an Asp Oδ can only pay for at most 3 freed waters).

## 7. Entropy contribution

The full free energy is

$$\Delta G_{\mathrm{bind}} = \Delta H_{\mathrm{bind}} - T \Delta S_{\mathrm{bind}}$$

with

$$\Delta S_{\mathrm{bind}} \;=\; \Delta S_{\mathrm{trans+rot}} \;+\; \Delta S_{\mathrm{side\text{-}chain}} \;+\; \Delta S_{\mathrm{backbone}} \;+\; \Delta S_{\mathrm{solvent}}$$

**We were not able to obtain a trustworthy estimate of ΔS in the time frame of this project.** Stub modules for all four terms exist under `entropy/` but each one has a known failure mode that we did not resolve (see the *What is not implemented* section of the README). Rather than report a number we don't believe, this section is intentionally left blank.

The cell below is a placeholder — fill it in once the entropy modules pass validation.

In [ ]:
# --- entropy estimate goes here ---
#
# Expected interface, once the modules are validated:
#
#   from total_entropy import compute_total_entropy
#   dS = compute_total_entropy(pdb_path, CHAIN_A, CHAIN_B, T=298.15,
#                              return_breakdown=True)
#
# Until then, leave dS undefined.

dS = None  # not implemented

## 8. Final ΔG

With ΔH in hand and ΔS missing, the headline ΔG is reported only as **ΔH alone**, with a clear marker that the entropy is unaccounted for. This is the most honest thing we can do — it lets the enthalpy be inspected and compared between complexes, without dressing up an incomplete pipeline as a complete one.

Once the entropy modules are validated, the cell below is the one place that needs to change.

In [ ]:
T  = 298.15  # K
dH = breakdown['TOTAL']

if dS is None:
    print(f'{PDB_ID} chains {CHAIN_A}/{CHAIN_B}:')
    print(f'   ΔH      = {dH:+.2f} kJ/mol')
    print(f'   -T·ΔS   = (not implemented)')
    print(f'   ΔG      = (not implemented)')
else:
    # Placeholder for when the entropy modules are wired in.
    # dS is expected in J/(mol·K) -> convert -T·dS to kJ/mol.
    minus_TdS = -T * dS / 1000.0
    dG = dH + minus_TdS
    print(f'{PDB_ID} chains {CHAIN_A}/{CHAIN_B}:')
    print(f'   ΔH      = {dH:+.2f} kJ/mol')
    print(f'   -T·ΔS   = {minus_TdS:+.2f} kJ/mol')
    print(f'   ΔG      = {dG:+.2f} kJ/mol')

## 9. Where to go from here

Concrete next steps, in roughly increasing order of effort:

1. **Calibrate the enthalpic well depths.** The constants in `pdb_utils_enthalpy.py` come from textbook ranges, not from a fit. A linear regression of (interaction counts, buried SASA) against experimental ΔG over a small set of well-characterised complexes (PDBbind subset) would give defensible weights.
2. **Validate the trans+rot module.** It already returns plausible numbers; the remaining work is to test it on a reference complex with a published estimate (e.g. Tidor–Karplus insulin dimer) and check the sign and order of magnitude.
3. **Replace the sphere-equivalent radius in the solvent module** with a proper SASA-based formulation.
4. **Tune the NMA cutoff and mode-matching threshold** for the backbone module against a benchmark complex.
5. **Add a small CLI** that wraps the four-line API into a single command, once the entropy side is trusted.